In [0]:
%run ../cralf_config/nb_00_config_lakeflow

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_users = spark.table(users_table)
df_products = spark.table(products_table)
df_orders = spark.table(orders_table)

In [0]:
df_enriched = (
    df_orders
        .join(df_users, on='user_id', how="left")
        .join(broadcast(df_products), on='product_id', how="left")
)

In [0]:
df_enriched = (
    df_enriched
        .withColumn('revenue', round(col('quantity') * col('price'), 2))
)

In [0]:
df_enriched = df_enriched.select(
    "order_id",
    "user_id",
    "product_id",
    "category",
    "signup_country",
    "quantity",
    "price",
    "revenue",
    "order_ts",
    "order_date"
)

In [0]:
df_enriched = (
    df_enriched
        .filter(col('price') > 0)
        .filter(col('quantity') > 0)
)

In [0]:
processing_date = processing_date

In [0]:
df_enriched = df_enriched.filter(col('order_date') == processing_date)

In [0]:
df_enriched.write.format("delta")\
    .mode("append")\
    .partitionBy("order_date")\
    .saveAsTable(enriched_orders_table)

In [0]:
display(
    spark.sql(f"SELECT * FROM {enriched_orders_table}")
)